# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [41]:
DECISION_DATE = "2026-02-28"

features = con.sql(f"""
    WITH feb AS (
        SELECT client_hash_id,
               content_hash_id,
               SUM(gsc_impressions) AS imp_feb,
               SUM(gsc_clicks) AS clicks_feb,
               SUM(gsc_sum_position) AS sum_pos_feb,
               COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_visible_feb,
               MIN(CASE WHEN gsc_impressions > 0 THEN gsc_avg_position END) AS best_pos_feb,
               SUM(CASE WHEN report_date <= DATE '2026-02-14' THEN gsc_impressions ELSE 0 END) AS imp_h1,
               SUM(CASE WHEN report_date >  DATE '2026-02-14' THEN gsc_impressions ELSE 0 END) AS imp_h2,
               MAX(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) AS has_ga4
        FROM {FEB}
        GROUP BY 1, 2
    ),
    mar AS (
        SELECT client_hash_id,
               content_hash_id,
               SUM(gsc_impressions) AS imp_mar
        FROM {MAR}
        GROUP BY 1, 2
    )
    SELECT
        f.client_hash_id,
        f.content_hash_id,

        -- Volume, February only
        f.imp_feb,
        f.clicks_feb,
        f.days_visible_feb,

        -- Rates, both inputs from February
        ROUND(f.clicks_feb::FLOAT / NULLIF(f.imp_feb, 0), 5) AS ctr_feb,
        ROUND(f.imp_feb::FLOAT / NULLIF(f.days_visible_feb, 0), 2) AS imp_per_active_day,

        -- Position: impression-weighted, lower is better
        ROUND(f.sum_pos_feb::FLOAT / NULLIF(f.imp_feb, 0), 2) AS pos_feb,
        COALESCE(f.best_pos_feb, 100) AS best_pos_feb,

        -- Within-February trend: second half against first half
        ROUND(f.imp_h2::FLOAT / NULLIF(f.imp_h1, 0), 3) AS trend_within_feb,

        -- GA4 presence as a flag, since only a small share of rows have GA4
        f.has_ga4,

        -- Content metadata, known before the decision date
        COALESCE(d.word_count, 0) AS word_count,
        CASE WHEN d.word_count IS NULL THEN 1 ELSE 0 END AS word_count_missing,
        COALESCE(d.search_volume, 0) AS search_volume,
        COALESCE(d.competition, 0) AS competition,
        COALESCE(d.backlinks, 0) AS backlinks,
        DATE_DIFF('day', d.content_created_date, DATE '{DECISION_DATE}') AS content_age_days,

        -- Categoricals, filled explicitly
        COALESCE(d.content_type, 'unknown') AS content_type,
        COALESCE(d.competition_level, 'unknown') AS competition_level,
        COALESCE(d.main_intent, 'unknown') AS main_intent,

        -- Label window: March only, never a feature
        COALESCE(m.imp_mar, 0) AS imp_mar

    FROM feb f
    LEFT JOIN mar m USING (client_hash_id, content_hash_id)
    LEFT JOIN {DIM_CONTENT} d USING (client_hash_id, content_hash_id)
    WHERE f.imp_feb >= 50
    ORDER BY f.client_hash_id, f.content_hash_id
""").df()

print(f"Rows: {len(features):,}")
print(f"Columns: {len(features.columns)}")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 93,654
Columns: 21


,client_hash_id,content_hash_id,imp_feb,clicks_feb,days_visible_feb,ctr_feb,imp_per_active_day,pos_feb,best_pos_feb,trend_within_feb,...,word_count,word_count_missing,search_volume,competition,backlinks,content_age_days,content_type,competition_level,main_intent,imp_mar
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,246.0,1.0,28,0.00407,8.79,17.24,8.50,0.662,...,3168,0,0,0.00,0,144,keyword article,LOW,informational,331.0
1,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,137.0,0.0,27,0.00000,5.07,9.66,7.75,1.076,...,4135,0,20,0.03,9,144,keyword article,LOW,commercial,33.0
2,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,121.0,0.0,28,0.00000,4.32,9.12,5.75,0.833,...,3211,0,0,0.00,0,144,keyword article,LOW,informational,145.0
3,client_0797ff3a1fc9a6a5,content_1207efddce873942,174.0,0.0,12,0.00000,14.50,13.38,9.00,NaN,...,3465,0,0,0.00,0,144,keyword article,LOW,informational,461.0
4,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,164.0,0.0,28,0.00000,5.86,11.48,7.00,0.451,...,3149,0,0,0.00,0,144,keyword article,LOW,informational,232.0


In [42]:
import numpy as np

df = features.copy()

# February has 28 days, March has 31 — compare daily rates, not raw sums
FEB_DAYS, MAR_DAYS = 28, 31
imp_rate_feb = df["imp_feb"] / FEB_DAYS
imp_rate_mar = df["imp_mar"] / MAR_DAYS

df["is_declining"] = (imp_rate_mar < 0.8 * imp_rate_feb).astype(int)

print(f"Positive rate: {df['is_declining'].mean():.1%} ({df['is_declining'].sum():,} declining)")
print(f"\nMissing values before fill:\n{df.isna().sum()[df.isna().sum() > 0]}")

# Remaining fills, stated explicitly rather than dropped
df["pos_feb"] = df["pos_feb"].fillna(100)
df["no_h1_impressions"] = df["trend_within_feb"].isna().astype(int)
df["trend_within_feb"] = df["trend_within_feb"].fillna(1.0)
df["content_age_days"] = df["content_age_days"].fillna(-1)

print(f"\nRows after fills: {len(df):,} (no rows dropped)")

Positive rate: 27.6% (25,887 declining)

Missing values before fill:
trend_within_feb    12369
dtype: int64

Rows after fills: 93,654 (no rows dropped)


In [43]:
import pandas as pd

# Remove encodings from a previous run so re-running this cell is idempotent
encoded_prefixes = ("ctype_", "intent_")
df = df.drop(columns=[c for c in df.columns if c.startswith(encoded_prefixes)], errors="ignore")
df = df.drop(columns=["competition_level_ord"], errors="ignore")

competition_order = {"unknown": 0, "LOW": 1, "MEDIUM": 2, "HIGH": 3}
df["competition_level_ord"] = df["competition_level"].map(competition_order).fillna(0).astype(int)

one_hot = pd.get_dummies(
    df[["content_type", "main_intent"]],
    prefix=["ctype", "intent"],
    dtype=int,
)
df = pd.concat([df, one_hot], axis=1)

drop_cols = ["client_hash_id", "content_hash_id", "imp_mar", "is_declining",
             "content_type", "competition_level", "main_intent"]
model_features = [c for c in df.columns if c not in drop_cols]

print(f"Model-ready features: {len(model_features)}")

Model-ready features: 25


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [44]:
summary = []
for c in feature_cols:
    summary.append({
        "feature": c,
        "dtype": str(df[c].dtype),
        "missing": int(df[c].isna().sum()),
        "unique": int(df[c].nunique()),
        "example": df[c].iloc[0],
    })

pd.DataFrame(summary)

,feature,dtype,missing,unique,example
0,imp_feb,float64,0,10891,246.0
1,clicks_feb,float64,0,347,1.0
2,days_visible_feb,int64,0,28,28
3,ctr_feb,float32,0,2348,0.00407
4,imp_per_active_day,float32,0,15899,8.79
5,pos_feb,float32,0,6162,17.24
6,best_pos_feb,float64,0,25525,8.5
7,trend_within_feb,float32,0,6521,0.662
8,has_ga4,int32,0,2,0
9,word_count,int64,0,6300,3168


**Decision moment: 2026-02-28.** Every feature below is measured on or before that
date. The label is measured over March, after it.

### Performance features (from `fact_content_daily_performance`, February partition)

| Feature | Meaning | Missing handling | Available when? |
|---|---|---|---|
| `imp_feb` | Total GSC impressions in February | None missing — pages need >= 50 to enter the frame | Yes, February only |
| `clicks_feb` | Total GSC clicks in February | None missing | Yes, February only |
| `days_visible_feb` | Count of February days with at least one impression | None missing | Yes, February only |
| `ctr_feb` | `clicks_feb / imp_feb` | Denominator cannot be 0 given the >= 50 filter | Yes, both inputs February |
| `imp_per_active_day` | `imp_feb / days_visible_feb` | Denominator cannot be 0 | Yes, both inputs February |
| `pos_feb` | Impression-weighted average GSC position | Filled with 100 where no position data | Yes, February only |
| `best_pos_feb` | Best single-day position reached in February | Filled with 100 where the page never ranked | Yes, February only |
| `trend_within_feb` | Second-half impressions divided by first-half. Intended as a momentum signal; tested in section 3 rather than assumed | Filled 1.0 (flat) for 12,369 pages with no first-half impressions, flagged separately | Yes, both halves inside February |
| `no_h1_impressions` | 1 when the page had no impressions in the first half of February | Constructed, never missing | Yes |
| `has_ga4` | 1 when GA4 data was available for the page in February | Constructed, never missing | Yes |

### Content metadata (from `dim_content`)

`dim_content` is a current-state snapshot, not history. I cannot verify what these
values were on 2026-02-28. All the fields below are slow-changing, so the risk is
low but not zero — this is stated rather than hidden.

| Feature | Meaning | Missing handling | Available when? |
|---|---|---|---|
| `word_count` | Article word count | Filled 0, with `word_count_missing` flag | Assumed stable, snapshot caveat above |
| `word_count_missing` | 1 when `word_count` was absent | Constructed, never missing | Yes |
| `search_volume` | Search-volume estimate for the target keyword | Filled 0 | Assumed stable, snapshot caveat |
| `competition` | Keyword competition score, 0-1 | Filled 0 | Assumed stable, snapshot caveat |
| `backlinks` | Backlink count for the page | Filled 0 | Assumed stable, snapshot caveat |
| `content_age_days` | Days from `content_created_date` to 2026-02-28 | Filled -1 where the creation date is absent | Yes — creation date cannot move |
| `content_type` | `keyword article` / `feedly article` / `comparison article` | Filled `unknown` | Assumed stable, snapshot caveat |
| `competition_level` | `LOW` / `MEDIUM` / `HIGH` | Filled `unknown` | Assumed stable, snapshot caveat |
| `main_intent` | `informational` / `commercial` / `transactional` / `navigational` | Filled `unknown` | Assumed stable, snapshot caveat |

### Encoding

The three categorical features are converted before modelling: `competition_level`
maps to an ordinal 0-3 because its levels have a natural order, while `content_type`
and `main_intent` become one-hot columns because theirs do not. The model receives
25 columns in total.

**One feature is weak by construction.** 91,629 of 93,654 pages (97.8%) are
`keyword article`, leaving 752 and 1,273 pages in the other two categories. A
near-constant column carries little information and gives a tree room to overfit
the small groups. I keep it for now and revisit it in section 3.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [45]:
correlations = df[model_features + ["is_declining"]].corr()["is_declining"].drop("is_declining")
correlations = correlations.reindex(correlations.abs().sort_values(ascending=False).index)

print("Correlation with the label, strongest first")
print(correlations.head(10).to_string())
print(f"\nMax absolute correlation: {correlations.abs().max():.3f}")

Correlation with the label, strongest first
no_h1_impressions       -0.124249
intent_unknown           0.101451
ctype_feedly article     0.083740
competition              0.073334
has_ga4                 -0.071051
ctr_feb                 -0.067282
word_count              -0.064360
intent_informational    -0.053162
word_count_missing       0.052703
ctype_keyword article   -0.045186

Max absolute correlation: 0.124


In [46]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X_tr, X_te, y_tr, y_te = train_test_split(
    df[model_features], df["is_declining"],
    test_size=0.3, random_state=42, stratify=df["is_declining"]
)

single = {}
for col in model_features:
    tree = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_tr[[col]], y_tr)
    single[col] = roc_auc_score(y_te, tree.predict_proba(X_te[[col]])[:, 1])

single = pd.Series(single).sort_values(ascending=False)
print("Single-feature ROC AUC, strongest first")
print(single.head(10).to_string())

suspects = single[single > 0.90]
print(f"\nFeatures scoring above 0.90 alone: {len(suspects)}")
if len(suspects):
    print(suspects.to_string())

Single-feature ROC AUC, strongest first
trend_within_feb      0.679314
days_visible_feb      0.636572
content_age_days      0.629638
word_count            0.608766
clicks_feb            0.583371
ctr_feb               0.579285
imp_per_active_day    0.578331
best_pos_feb          0.573132
imp_feb               0.560440
pos_feb               0.551318

Features scoring above 0.90 alone: 0


In [47]:
def auc_with(cols):
    from sklearn.ensemble import RandomForestClassifier
    model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_tr[cols], y_tr)
    return roc_auc_score(y_te, model.predict_proba(X_te[cols])[:, 1])


auc_clean = auc_with(model_features)

# Deliberately plant a leak: March impressions are the label's own ingredient
X_tr_leak = X_tr.copy()
X_te_leak = X_te.copy()
X_tr_leak["imp_mar"] = df.loc[X_tr.index, "imp_mar"]
X_te_leak["imp_mar"] = df.loc[X_te.index, "imp_mar"]

from sklearn.ensemble import RandomForestClassifier
leaky_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
leaky_model.fit(X_tr_leak, y_tr)
auc_leaky = roc_auc_score(y_te, leaky_model.predict_proba(X_te_leak)[:, 1])

print(f"Clean features        : ROC AUC {auc_clean:.3f}")
print(f"With imp_mar planted  : ROC AUC {auc_leaky:.3f}")

leak_single = DecisionTreeClassifier(max_depth=4, random_state=42)
leak_single.fit(df.loc[X_tr.index, ["imp_mar"]], y_tr)
leak_auc = roc_auc_score(y_te, leak_single.predict_proba(df.loc[X_te.index, ["imp_mar"]])[:, 1])
print(f"\nimp_mar alone         : ROC AUC {leak_auc:.3f}")


Clean features        : ROC AUC 0.802
With imp_mar planted  : ROC AUC 0.996

imp_mar alone         : ROC AUC 0.767


**The single-feature detector has a blind spot.** `imp_mar` — the label's own
ingredient — scores only 0.767 alone, below my 0.90 threshold. It would have passed
the cell-2 screen undetected. Yet added to the honest features it lifts ROC AUC
from 0.802 to 0.996.

The reason is that `is_declining` is a *ratio* test: March impressions against
February impressions. March alone is ambiguous — 500 impressions is a collapse for
a large page and growth for a small one. The leak only becomes visible once the
February baseline is present alongside it.

Directional implication: single-feature screening is necessary but not sufficient.
A leaked column that only works in combination with legitimate features will pass
it. The window audit in the next cell is the check that would have caught this one,
because it tests provenance rather than predictive strength.

In [48]:
FEATURE_WINDOW_END = "2026-02-28"

audit = {
    "imp_feb": "2026-02", "clicks_feb": "2026-02", "days_visible_feb": "2026-02",
    "ctr_feb": "2026-02", "imp_per_active_day": "2026-02", "pos_feb": "2026-02",
    "best_pos_feb": "2026-02", "trend_within_feb": "2026-02",
    "no_h1_impressions": "2026-02", "has_ga4": "2026-02",
    "word_count": "dim_content snapshot", "word_count_missing": "dim_content snapshot",
    "search_volume": "dim_content snapshot", "competition": "dim_content snapshot",
    "backlinks": "dim_content snapshot", "content_age_days": "dim_content snapshot",
    "competition_level_ord": "dim_content snapshot",
}
for col in model_features:
    if col.startswith(("ctype_", "intent_")):
        audit[col] = "dim_content snapshot"

missing = [c for c in model_features if c not in audit]
march_sourced = [c for c, src in audit.items() if src == "2026-03"]

print(f"Features audited      : {len(model_features)}")
print(f"Undocumented source   : {len(missing)} {missing if missing else ''}")
print(f"Sourced from March    : {len(march_sourced)}")
assert not missing, "Every feature must have a documented source window"
assert not march_sourced, "No feature may come from the label window"
print("\nAudit passed: no feature draws on the March label window.")

Features audited      : 25
Undocumented source   : 0 
Sourced from March    : 0

Audit passed: no feature draws on the March label window.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Every field below exists in the warehouse and was available to me. I refused each
one for a stated reason.

### Excluded because it falls inside the label window

| Field | Why |
|---|---|
| `imp_mar` and every March column | March is the outcome window. Section 3 shows that adding `imp_mar` alone lifts ROC AUC from 0.802 to 0.996 — the model reads the answer rather than predicting it |

### Excluded because `dim_content` is a snapshot, not history

| Field | Why |
|---|---|
| `content_updated_date` | Records the most recent update at export time. A page updated in March would carry a March date, so this field can encode the label window without looking like it does |
| `last_optimized_date` | Same problem, and it is a direct trace of FlyRank's own editorial workflow |
| `optimization_eligible_date` | Derived from FlyRank's optimization logic, so a model using it would relearn an internal product rule rather than a search signal |

These three are the most dangerous fields in the release: they look like ordinary
metadata and would raise no suspicion in a feature list. I keep the slow-changing
fields from the same table (`word_count`, `search_volume`, `competition`,
`backlinks`, `content_created_date`) and state the snapshot caveat in section 2
rather than pretending it does not apply.

### Excluded because they identify rather than describe

| Field | Why |
|---|---|
| `client_hash_id` | A model could memorise which clients decline instead of learning why pages decline. Kept for grouping and splitting only |
| `content_hash_id` | Unique per row, so it carries no generalisable signal |
| `keyword_hash_id`, `url_hash_id` | Same reason; pseudonymous identifiers, not measurements |

### Excluded because coverage is too thin to use

| Field | Why |
|---|---|
| All `ga4_*` columns (`ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`, `ga4_pageviews`, `scroll_events`) | Measured in w03: only 4.2% of March rows have `ga4_data_available IS TRUE`. A feature missing for ~96% of pages would mostly encode which clients have GA4 configured. I keep `has_ga4` as a binary flag instead |
| `sessions_ai` and the per-provider AI columns (`ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other`) | GA4-derived, so they inherit the same 4.2% coverage problem |
| `sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`, `sessions_paid` | Same GA4 coverage limit |

### Excluded because they are status flags, not signals

| Field | Why |
|---|---|
| `is_published`, `is_deleted` | Current state at export time, not February state. A page deleted in March would appear deleted in my feature window |
| `client_has_gsc`, `client_has_ga4`, `gsc_data_available` | Describe data collection, not page performance. Using them would let the model separate clients by instrumentation rather than by content |

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.